# Imports

In [31]:
from pathlib import Path
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Load the resolved event dataset

In [32]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "processed" / "spy_event_dataset.csv"

df = pd.read_csv(data_path)
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
df = df.sort_values("Datetime").reset_index(drop=True)

df.shape

(74, 40)

# Inspect the dataset and class balance

In [33]:
print(df["label_continuation"].value_counts())
print()
print(df["label_continuation"].value_counts(normalize=True))

label_continuation
1    38
0    36
Name: count, dtype: int64

label_continuation
1    0.513514
0    0.486486
Name: proportion, dtype: float64


# Define the non-leaky feature set and target

In [34]:
feature_cols = [
    "ret_k",
    "sigma_k",
    "event_zscore",
    "event_size",
    "vol_k_rel",
    "range_k",
    "vol_regime",
    "prev_ret_k",
    "mins_from_open",
    "ret_2",
    "ret_3",
    "ret_1_lag1",
    "ret_1_lag2",
    "ret_1_lag3",
    "ret_1_mean_3",
    "ret_1_std_3",
    "up_frac_3",
    "down_frac_3",
    "max_ret_1_3",
    "min_ret_1_3",
    "body",
    "bar_range",
    "upper_wick",
    "lower_wick",
    "vol_1_rel",
    "vol_mean_3_rel",
]

target_col = "label_continuation"

X = df[feature_cols].copy()
y = df[target_col].copy()

X.head()

,ret_k,sigma_k,event_zscore,event_size,vol_k_rel,range_k,vol_regime,prev_ret_k,mins_from_open,ret_2,...,up_frac_3,down_frac_3,max_ret_1_3,min_ret_1_3,body,bar_range,upper_wick,lower_wick,vol_1_rel,vol_mean_3_rel
0,-0.001182,0.000527,2.244506,0.001182,1.305269,0.001849,0.000378,0.000489,598,-0.001386,...,0.333333,0.666667,0.000204,-0.001073,-0.000307,0.000877,0.000219,0.000351,1.408860,1.357037
1,-0.002022,0.000805,2.512689,0.002022,1.128674,0.002084,0.000592,0.000322,654,-0.001583,...,0.000000,1.000000,-0.000440,-0.001144,-0.001129,0.001350,0.000176,0.000044,1.608333,1.104074
2,0.002690,0.001270,2.119045,0.002690,1.334579,0.003588,0.000744,-0.000088,532,0.002617,...,1.000000,0.000000,0.001543,0.000073,0.001057,0.001732,0.000675,0.000000,2.364301,1.302594
3,-0.000915,0.000413,2.215032,0.000915,1.815374,0.001481,0.000290,-0.000593,662,-0.001010,...,0.333333,0.666667,0.000095,-0.000637,-0.000366,0.000733,0.000308,0.000059,1.716372,1.742940
4,0.001396,0.000501,2.787006,0.001396,1.294525,0.001673,0.000344,-0.000378,454,0.001149,...,1.000000,0.000000,0.001061,0.000087,0.001047,0.001200,0.000058,0.000095,1.459550,1.286112


# Split the events chronologically into train and test sets

In [35]:
split_idx = int(len(df) * 0.7)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("train size:", len(X_train))
print("test size:", len(X_test))
print()
print("train class balance:")
print(y_train.value_counts())
print()
print("test class balance:")
print(y_test.value_counts())

train size: 51
test size: 23

train class balance:
label_continuation
1    26
0    25
Name: count, dtype: int64

test class balance:
label_continuation
1    12
0    11
Name: count, dtype: int64


# Fit a logistic regression baseline

In [36]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work

# Generate predictions on the test set

In [37]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

majority_class = y_train.mode().iloc[0]
y_pred_baseline = pd.Series(majority_class, index=y_test.index)

In [38]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

majority_class = y_train.mode().iloc[0]
y_pred_baseline = pd.Series(majority_class, index=y_test.index)

# Evaluate the model against a naive majority-class baseline

In [39]:
print("model accuracy:", accuracy_score(y_test, y_pred))
print("baseline accuracy:", accuracy_score(y_test, y_pred_baseline))
print()
print("confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("classification report:")
print(classification_report(y_test, y_pred, digits=4))

if y_test.nunique() == 2:
    print("roc auc:", roc_auc_score(y_test, y_prob))

model accuracy: 0.6086956521739131
baseline accuracy: 0.5217391304347826

confusion matrix:
[[6 5]
 [4 8]]

classification report:
              precision    recall  f1-score   support

           0     0.6000    0.5455    0.5714        11
           1     0.6154    0.6667    0.6400        12

    accuracy                         0.6087        23
   macro avg     0.6077    0.6061    0.6057        23
weighted avg     0.6080    0.6087    0.6072        23

roc auc: 0.48484848484848486


# Inspect the learned logistic regression coefficients

In [40]:
coef = pd.Series(
    model.named_steps["clf"].coef_[0],
    index=feature_cols
).sort_values(key=lambda s: s.abs(), ascending=False)

coef

vol_regime        0.813706
prev_ret_k        0.773146
vol_mean_3_rel    0.564937
vol_k_rel         0.540918
up_frac_3        -0.504888
ret_1_lag1        0.486537
min_ret_1_3      -0.446777
vol_1_rel         0.443459
event_size       -0.407588
range_k           0.402525
max_ret_1_3      -0.335494
sigma_k          -0.328419
upper_wick       -0.313927
lower_wick       -0.279394
down_frac_3      -0.245713
bar_range        -0.236655
ret_2             0.178414
ret_1_std_3      -0.173219
ret_1_lag2       -0.164881
ret_3             0.102524
ret_k             0.102524
ret_1_mean_3      0.102366
mins_from_open   -0.084547
event_zscore      0.045465
ret_1_lag3       -0.043013
body             -0.040821
dtype: float64

# Fit a random forest classifier

In [41]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=3,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",3
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

# Generate random forest predictions

In [42]:
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate the random forest against the same naive baseline

In [43]:
print("random forest accuracy:", accuracy_score(y_test, rf_pred))
print("baseline accuracy:", accuracy_score(y_test, y_pred_baseline))
print()

print("confusion matrix:")
print(confusion_matrix(y_test, rf_pred))
print()

print("classification report:")
print(classification_report(y_test, rf_pred, digits=4))

if y_test.nunique() == 2:
    print("roc auc:", roc_auc_score(y_test, rf_prob))

random forest accuracy: 0.5217391304347826
baseline accuracy: 0.5217391304347826

confusion matrix:
[[4 7]
 [4 8]]

classification report:
              precision    recall  f1-score   support

           0     0.5000    0.3636    0.4211        11
           1     0.5333    0.6667    0.5926        12

    accuracy                         0.5217        23
   macro avg     0.5167    0.5152    0.5068        23
weighted avg     0.5174    0.5217    0.5106        23

roc auc: 0.4469696969696969


# Compare class balance in train vs test

In [44]:
print("train:")
print(y_train.value_counts(normalize=True))
print()
print("test:")
print(y_test.value_counts(normalize=True))

train:
label_continuation
1    0.509804
0    0.490196
Name: proportion, dtype: float64

test:
label_continuation
1    0.521739
0    0.478261
Name: proportion, dtype: float64


# Do features encode any Signal?

In [45]:
for col in feature_cols:
    X1_train = X_train[[col]]
    X1_test = X_test[[col]]

    model_1 = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000))
    ])

    model_1.fit(X1_train, y_train)
    pred_1 = model_1.predict(X1_test)

    print(col, accuracy_score(y_test, pred_1))

ret_k 0.5652173913043478
sigma_k 0.5652173913043478
event_zscore 0.6086956521739131
event_size 0.5217391304347826
vol_k_rel 0.5652173913043478
range_k 0.34782608695652173
vol_regime 0.391304347826087
prev_ret_k 0.5652173913043478
mins_from_open 0.6086956521739131
ret_2 0.5652173913043478
ret_3 0.5652173913043478
ret_1_lag1 0.5217391304347826
ret_1_lag2 0.5652173913043478
ret_1_lag3 0.43478260869565216
ret_1_mean_3 0.5652173913043478
ret_1_std_3 0.43478260869565216
up_frac_3 0.5652173913043478
down_frac_3 0.5652173913043478
max_ret_1_3 0.5652173913043478
min_ret_1_3 0.6086956521739131
body 0.5652173913043478
bar_range 0.391304347826087
upper_wick 0.43478260869565216
lower_wick 0.34782608695652173
vol_1_rel 0.4782608695652174
vol_mean_3_rel 0.5652173913043478


# Drop range

In [27]:
feature_cols_2 = [
    "ret_k",
    "sigma_k",
    "event_zscore",
    "event_size",
    "vol_k_rel",
    "vol_regime",
    "prev_ret_k",
    "mins_from_open",
]

X2 = df[feature_cols_2].copy()

X2_train = X2.iloc[:split_idx]
X2_test = X2.iloc[split_idx:]

# from sklearn.ensemble import HistGradientBoostingClassifier

In [46]:
gb_model = HistGradientBoostingClassifier(
    max_depth=3,
    learning_rate=0.05,
    max_iter=200,
    min_samples_leaf=5,
    random_state=42
)

gb_model.fit(X_train, y_train)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",3
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",5
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dtype

# Generate gradient boosting predictions

In [47]:
gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

# Evaluate the gradient boosting model

In [48]:
print("gradient boosting accuracy:", accuracy_score(y_test, gb_pred))
print("baseline accuracy:", accuracy_score(y_test, y_pred_baseline))
print()

print("confusion matrix:")
print(confusion_matrix(y_test, gb_pred))
print()

print("classification report:")
print(classification_report(y_test, gb_pred, digits=4))

if y_test.nunique() == 2:
    print("roc auc:", roc_auc_score(y_test, gb_prob))

gradient boosting accuracy: 0.43478260869565216
baseline accuracy: 0.5217391304347826

confusion matrix:
[[5 6]
 [7 5]]

classification report:
              precision    recall  f1-score   support

           0     0.4167    0.4545    0.4348        11
           1     0.4545    0.4167    0.4348        12

    accuracy                         0.4348        23
   macro avg     0.4356    0.4356    0.4348        23
weighted avg     0.4364    0.4348    0.4348        23

roc auc: 0.4545454545454546
